# KRM-Core: BPE + RWKV HuggingFace Training

BPE tokenizer (32K vocab) + RWKV model with Kavramsal Parçalı Eğitim.

**Veri setleri:**
- `HuggingFaceFW/fineweb` (15T token, genel)
- `HuggingFaceFW/fineweb-edu` (1.3T token, eğitim)
- `bigcode/the-stack-v2` (900B token, kod)
- `allenai/c4` (temiz web)

**Modeller:** rwkv_10m (36M), rwkv_50m (74M), rwkv_200m (252M), rwkv_1b (780M), rwkv_7b (7.25B)

In [ ]:
# Bağımlılıklar
!pip install torch numpy tqdm datasets tokenizers huggingface_hub

import torch, gc, json, time, os, random, sys, math
from pathlib import Path
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# RWKV MODEL (KRM-Core torch_backend'den)
# ============================================================

class WKVMemory(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.log_gain = nn.Parameter(torch.zeros(d_model))
        self.log_decay = nn.Parameter(torch.zeros(d_model))

    def forward(self, values, keys):
        B, S, D = values.shape
        gain = torch.exp(self.log_gain)
        decay = torch.exp(-torch.exp(self.log_decay))
        out = torch.zeros_like(values)
        s = torch.zeros(B, D, device=values.device)
        n = torch.zeros(B, D, device=values.device)
        for t in range(S):
            k = keys[:, t] * gain
            s = s * decay + k * values[:, t]
            n = n * decay + k
            out[:, t] = s / (n + 1e-8)
        return out


class RWKVBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.time_k = nn.Linear(d_model, d_model, bias=False)
        self.time_v = nn.Linear(d_model, d_model, bias=False)
        self.time_r = nn.Linear(d_model, d_model, bias=False)
        self.time_o = nn.Linear(d_model, d_model, bias=False)
        self.wkv = WKVMemory(d_model)
        self.chan_k = nn.Linear(d_model, d_ff, bias=False)
        self.chan_v = nn.Linear(d_ff, d_model, bias=False)
        self.chan_r = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        r = x
        h = self.ln1(x)
        k, v, rg = self.time_k(h), self.time_v(h), self.time_r(h)
        h = r + torch.sigmoid(rg) * self.time_o(self.wkv(v, k))
        r = h
        h = self.ln2(h)
        k = F.relu(self.chan_k(h)) ** 2
        h = r + torch.sigmoid(self.chan_r(h)) * self.chan_v(k)
        return h


class RWKVLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_ff):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.embed(input_ids)
        for b in self.blocks:
            x = b(x)
        return self.head(self.norm(x))

In [ ]:
# ============================================================
# BPE TOKENIZER (KRM-Core bpe_tokenizer'dan)
# ============================================================

# Colab'da tokenizer eğit veya GitHub'dan yükle
import json
from tokenizers import Tokenizer, AddedToken, trainers, pre_tokenizers, normalizers, decoders, processors
from tokenizers.models import BPE

SPECIAL_TOKENS = [
    '<|SOURCE|>', '<|QUERY|>', '<|CONTEXT|>', '<|CONCEPTS|>', '<|EDGES|>',
    '<|HOT_GRAPH|>', '<|POLICY|>', '<|ANSWER_PLAN|>', '<|ANSWER|>', '<|INTENT|>',
    '<|DOMAIN|>', '<|SHARD|>', '<|UNCERTAIN|>', '<|SPECULATIVE|>', '<|DO_NOT_CLAIM|>',
    '<|SUPPORTED_BY_SOURCE|>', '<|MISSING_INFO|>', '<|END|>',
    '<|PAD|>', '<|UNK|>', '<|BOS|>', '<|EOS|>'
]

def train_bpe_tokenizer(corpus_path, vocab_size=32768):
    tokenizer = Tokenizer(BPE(unk_token='<|UNK|>'))
    tokenizer.normalizer = normalizers.NFKC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

    special = [AddedToken(t, single_word=False, normalized=False) for t in SPECIAL_TOKENS]
    tokenizer.add_special_tokens(special)

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=[str(t) for t in special],
        show_progress=True,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    tokenizer.train([corpus_path], trainer)
    return tokenizer

class BPETokenizer:
    def __init__(self, tokenizer):
        self._tokenizer = tokenizer
        self.vocab_size = tokenizer.get_vocab_size()
    def encode(self, text):
        return self._tokenizer.encode(text).ids
    def decode(self, ids):
        return self._tokenizer.decode(ids)

print("BPE tokenizer yükleniyor...")

In [ ]:
# ============================================================
# MODEL KONFIGURASYONU
# ============================================================

MODEL_CONFIGS = {
    'rwkv_10m':  {'d_model': 384,  'n_layers': 6,  'd_ff': 1536,  'desc': '36M params'},
    'rwkv_50m':  {'d_model': 640,  'n_layers': 12, 'd_ff': 2560,  'desc': '74M params'},
    'rwkv_200m': {'d_model': 1024, 'n_layers': 16, 'd_ff': 4096,  'desc': '252M params'},
    'rwkv_1b':   {'d_model': 1536, 'n_layers': 24, 'd_ff': 6144,  'desc': '780M params'},
    'rwkv_7b':   {'d_model': 4096, 'n_layers': 32, 'd_ff': 16384, 'desc': '7.25B params'},
}

# --- AYARLAR ---
MODEL = 'rwkv_10m'     # rwkv_10m, rwkv_50m, rwkv_200m, rwkv_1b, rwkv_7b
DATASET = 'fineweb'     # fineweb, fineweb_edu, the_stack_v2, c4
BATCH_SIZE = 8          # GPU VRAM'e gore ayarla (T4: 8-16, A100: 32-64)
SEQ_LEN = 512           # Dizi uzunluğu
MAX_STEPS = 1000        # Adim sayisi
LR = 3e-4               # Öğrenme hızı
SAMPLE_SIZE = 10000     # Kaç örnek (streaming=False için)

cfg = MODEL_CONFIGS[MODEL]
print(f"Model: {MODEL} ({cfg['desc']})")
print(f"Dataset: {DATASET}")
print(f"Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}, Steps: {MAX_STEPS}")

In [ ]:
# ============================================================
# HUGGINGFACE VERI SETI YUKLE
# ============================================================

ds_map = {
    'fineweb': ('HuggingFaceFW/fineweb', 'sample-10BT'),
    'fineweb_edu': ('HuggingFaceFW/fineweb-edu', 'sample-10BT'),
    'the_stack_v2': ('bigcode/the-stack-v2', None),
    'c4': ('allenai/c4', 'en'),
}

path, config = ds_map[DATASET]
print(f"Yükleniyor: {path} (config={config})")

ds = load_dataset(path, config, split='train', streaming=True)

# İlk SAMPLE_SIZE örneği al ve kendi corpus'unu oluştur
texts = []
for i, example in enumerate(ds):
    if i >= SAMPLE_SIZE:
        break
    text = example.get('text', example.get('content', ''))
    if text:
        texts.append(text.strip())

corpus = '\n\n---\n\n'.join(texts)
print(f"Corpus: {len(corpus):,} chars ({len(texts):,} docs)")

# Cache'e yaz
Path('hf_cache').mkdir(exist_ok=True)
with open('hf_cache/corpus.txt', 'w', encoding='utf-8') as f:
    f.write(corpus)

In [ ]:
# ============================================================
# BPE TOKENIZER EGIT
# ============================================================

print("BPE tokenizer eğitiliyor... (vocab=32K)")
t = train_bpe_tokenizer('hf_cache/corpus.txt', vocab_size=32768)
tokenizer = BPETokenizer(t)
print(f"Vocab: {tokenizer.vocab_size:,}")

# Test
sample = "Hello world, KRM-Core RWKV training with BPE tokenizer!"
enc = tokenizer.encode(sample)
dec = tokenizer.decode(enc)
print(f"Test: '{sample}' -> {len(enc)} tokens (compression: {len(sample.encode())/len(enc):.1f}x)")

# Tokenize corpus
all_ids = tokenizer.encode(corpus)
print(f"Corpus tokenized: {len(all_ids):,} tokens")
del corpus  # free memory
gc.collect()

In [ ]:
# ============================================================
# EGITIM
# ============================================================

class TokenDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.seq_len = seq_len
        n = (len(tokens) - 1) // seq_len
        self.data = torch.tensor(tokens[:n * seq_len + 1], dtype=torch.long)
        print(f"  {n} chunks of {seq_len}")
    def __len__(self):
        return (len(self.data) - 1) // self.seq_len
    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.data[start:start + self.seq_len]
        y = self.data[start + 1:start + self.seq_len + 1]
        return x, y

dataset = TokenDataset(all_ids, SEQ_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
del all_ids
gc.collect()

# Model
model = RWKVLM(
    vocab_size=tokenizer.vocab_size,
    d_model=cfg['d_model'],
    n_layers=cfg['n_layers'],
    d_ff=cfg['d_ff'],
).to(device)

params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {params:,} params ({params*4/1024/1024:.1f} MB FP32)")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, MAX_STEPS)

history = {'loss': [], 'ppl': []}
best_loss = float('inf')
step = 0

print("\n" + "="*50)
print("TRAINING")
print("="*50)

model.train()
t0 = time.time()

while step < MAX_STEPS:
    for x, y in loader:
        if step >= MAX_STEPS:
            break
        step += 1

        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history['loss'].append(loss.item())
        history['ppl'].append(math.exp(loss.item()))

        if step % 10 == 0:
            elapsed = time.time() - t0
            avg_loss = sum(history['loss'][-10:]) / 10
            avg_ppl = sum(history['ppl'][-10:]) / 10
            print(f"Step {step:5d}/{MAX_STEPS} | Loss: {avg_loss:.4f} | PPL: {avg_ppl:.2f} | {elapsed:.1f}s")

        if step % 100 == 0:
            # Model kaydet
            path = f'rwkv_{MODEL}_step{step}.pt'
            torch.save({
                'model_state': model.state_dict(),
                'config': cfg,
                'step': step,
                'loss': loss.item(),
                'vocab_size': tokenizer.vocab_size,
            }, path)
            if loss.item() < best_loss:
                best_loss = loss.item()
                torch.save(model.state_dict(), f'rwkv_{MODEL}_best.pt')
                print(f"  [*] Best model saved!")

print(f"\n{'='*50}")
print(f"TAMAMLANDI! {step} steps, {time.time()-t0:.1f}s")
print(f"Best loss: {best_loss:.4f} (PPL: {math.exp(best_loss):.2f})")
print(f"Model: {params:,} params")

In [ ]:
# ============================================================
# ORNEK URETIM
# ============================================================

model.eval()
prompt = "KRM-Core is a"
input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

with torch.no_grad():
    for _ in range(200):
        logits = model(input_ids[:, -SEQ_LEN:])
        probs = F.softmax(logits[0, -1] / 0.8, dim=0)
        next_id = torch.multinomial(probs, 1).item()
        input_ids = torch.cat([input_ids, torch.tensor([[next_id]], device=device)], dim=1)

output = tokenizer.decode(input_ids[0].tolist())
print("--- GENERATED ---")
print(output[:1000])

In [ ]:
# ============================================================
# DRIVE'A KAYDET
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

save_dir = Path(f'/content/drive/MyDrive/KRM-Core/{MODEL}')
save_dir.mkdir(parents=True, exist_ok=True)

# Son model
torch.save({
    'model_state': model.state_dict(),
    'config': cfg,
    'history': history,
    'best_loss': best_loss,
    'params': params,
    'dataset': DATASET,
    'vocab_size': tokenizer.vocab_size,
}, save_dir / 'final_model.pt')

# Tokenizer
t.save(str(save_dir / 'tokenizer.json'))

# Config
with open(save_dir / 'config.json', 'w') as f:
    json.dump({'model': MODEL, **cfg, 'dataset': DATASET, 'vocab_size': tokenizer.vocab_size}, f, indent=2)

print(f"Kaydedildi: {save_dir}/")
for f in save_dir.glob('*'):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f}MB)")

## Çoklu Veri Seti / Blok Eğitimi

Kavramsal Parçalı Eğitim ile her veri seti türü ayrı blok:
- Blok 1: FineWeb-Edu (eğitim/reasoning)
- Blok 2: The Stack v2 (kod)
- Blok 3: DCLM-Baseline (genel bilgi)

Her blok ayrı eğitilir, inference'da Rezonans Motoru birleştirir.